# 실습 - 이미지 설명 생성

**이미지의 내용을 문장으로 서술**

**작업 개요**

- 작업 이름 — `image-to-text`
- 입력 — 이미지 한 장
- 출력 — 생성된 문장 (`generated_text`)
- 라벨이 아닌 **자유 형식 서술** — 정해진 범주 없음
- 생성 방식 — 실행할 때마다 표현이 달라질 수 있음

**앞선 작업과의 차이**

| 구분 | 출력 | 범주 |
|---|---|---|
| 분류 | 라벨 · 점수 | 정해져 있음 |
| 탐지 | 라벨 · 점수 · 상자 | 정해져 있음 |
| 세그멘테이션 | 마스크 | 없음 |
| 깊이 추정 | 깊이 값 | 없음 |
| **설명 생성** | **문장** | **정해져 있지 않음** |

**데이터**

- 실내외 일상 장면 이미지 — 깊이 추정과 동일 데이터
- 앞선 실습의 전문 영상(궤도 · 혈액 · 기판)도 함께 입력

**모델**

- 이미지 인코더 + 문장 생성기 결합 구조
- 일상 사진으로 학습 — 서술 범위가 그에 좌우

**실습 순서**

- 환경 준비
- 데이터 불러오기
- 모델 준비 · 추론 실행
- 여러 장면 서술 비교
- 전문 영상에 적용 — 한계 확인

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import torch

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- 실내외 일상 장면 이미지
- `split="train[:50]"` — 앞 50장만 사용
- 깊이 추정 실습과 **동일 데이터** — 이미 받아둔 경우 즉시 완료

In [ ]:
ds = load_dataset(
    "merve/scene_parse_150",
    split="train[:50]"
)

image = ds[0]["image"]
print("이미지 크기:", image.size)
image

---
## 2. 모델 준비

- `task="image-to-text"` — 이미지 설명 생성 작업
- `model` — 이미지 인코더와 문장 생성기를 결합한 모델
- `device=0` — GPU에서 실행

첫 실행 시 모델 내려받기로 시간이 소요된다.

In [ ]:
cap = pipeline(
    task="image-to-text",
    model="nlpconnect/vit-gpt2-image-captioning",
    device=0
)

print("모델 준비 완료")

---
## 3. 추론 실행

- 출력은 목록 형태 — `generated_text` 항목에 문장이 담김
- 라벨이나 점수가 없음 — **문장 하나만** 반환

In [ ]:
result = cap(image)

print(result)
print()
print("서술:", result[0]["generated_text"])

### 3-1. 여러 장면 서술 비교

- 장면에 따라 서술이 어떻게 달라지는지 관찰
- 사물이 뚜렷한 장면 · 복잡한 장면 비교

In [ ]:
for n in range(5):
    img = ds[n]["image"]
    text = cap(img)[0]["generated_text"]
    print(f"[{n}] {text}")
    display(img.resize((240, 180)))

### 3-2. 같은 이미지 반복 실행

- 생성 방식이므로 실행할 때마다 표현이 달라질 수 있음
- 분류·탐지와 달리 **출력이 고정되지 않음**

In [ ]:
for i in range(3):
    print(f"{i+1}회 —", cap(image)[0]["generated_text"])

---
## 4. 전문 영상에 적용

- 앞선 실습에서 사용한 이미지에 동일 모델 적용
- 학습 데이터(일상 사진)와 성격이 다른 경우 결과 확인

**예상**

- 일상 장면 — 구체적이고 정확한 서술
- 전문 영상 — 색 · 질감 수준의 서술에 그침

In [ ]:
# 앞선 실습 데이터 불러오기
ds_rail = load_dataset(
    "crangana/railroad-fault-detection", split="train[:5]")
ds_blood = load_dataset(
    "keremberke/blood-cell-object-detection", name="full", split="train[:5]")
ds_pcb = load_dataset(
    "keremberke/pcb-defect-segmentation", name="full", split="train[:5]")

samples = [
    ("일반 장면", ds[0]["image"]),
    ("철도 궤도", ds_rail[0]["image"]),
    ("혈액 도말", ds_blood[0]["image"]),
    ("PCB 기판", ds_pcb[0]["image"]),
]

print("준비 완료")

In [ ]:
for name, img in samples:
    text = cap(img)[0]["generated_text"]
    print(f"[{name}] {text}")
    display(img.resize((240, 180)))
    print()

**확인 사항**

- 일상 장면 — 사물과 배치를 구체적으로 서술
- 전문 영상 — 색 · 질감 수준에 그치거나 무관한 서술
- 원인 — 학습 데이터(일상 사진)의 범위

**모델은 항상 문장을 생성한다. 정확하다는 뜻은 아니다.**

---
## 5. 다른 장면 확인

- 이미지 번호를 바꾸어 다른 장면 확인
- 웹 이미지 주소로도 실행 가능 — `cap("이미지 주소")`

In [ ]:
n = 10
img = ds[n]["image"]

print(cap(img)[0]["generated_text"])
img

---
## 6. 정리

**수행 내용**

- 이미지 한 장 → 자유 형식 문장 생성
- 여러 장면의 서술 비교
- 전문 영상에 적용하여 한계 확인

**앞선 작업과의 차이**

- 출력이 **라벨이 아닌 문장** — 정해진 범주 없음
- **생성 방식** — 실행할 때마다 표현이 달라질 수 있음
- 점수가 없어 확신 정도를 알 수 없음

**한계**

- 학습 데이터(일상 사진)의 범위가 서술 능력을 결정
- 전문 영상은 색 · 질감 수준의 서술에 그침
- 모델은 항상 문장을 생성 — 정확하다는 뜻은 아님

**연결**

- 자기 분야에 맞는 서술이 필요하면 → 해당 분야 데이터로 미세조정
- 특정 항목을 묻고 싶으면 → 질의응답 작업(⑥ · ⑦)

---

데이터 출처: merve/scene_parse_150 (Hugging Face Hub 공개)